### 0. Setup

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import ensure_runtime
ensure_runtime(ROOT)

PosixPath('/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs')

In [ ]:
from src.spark import create_spark
from src.lake import GOLD, SILVER, read_delta

spark = create_spark("ml-pipeline")

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    SQLTransformer,
    StandardScaler,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.regression import GBTRegressor
from pyspark.sql import functions as F
from src.lake import GOLD, SILVER, read_delta

:: loading settings :: url = jar:file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/samuelflodin/.ivy2/cache
The jars for the packages stored in: /Users/samuelflodin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d4c823e7-4e23-46ca-80a7-d4fc2d913294;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 90ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |

### 1. Load Data & Split

In [ ]:
df_integrated = read_delta(spark, GOLD / "integrated_taxi_trips")

df_clean = df_integrated.filter(
    (F.col("fare_amount") > 0) & 
    (F.col("trip_distance") > 0.05) & 
    (~F.col("pickup_borough").isin("Unknown", "EWR"))
)

train_raw = df_clean.filter("pickup_date < '2024-02-21'")
val_raw = df_clean.filter("pickup_date >= '2024-02-21' AND pickup_date < '2024-03-11'")
test_raw = df_clean.filter("pickup_date >= '2024-03-11'")

print(f"Training dataset record count:   {train_raw.count():,}")
print(f"Validation dataset record count: {val_raw.count():,}")
print(f"Test dataset record count:       {test_raw.count():,}")

26/10/01 11:10:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training dataset record count:   4,801,899
Validation dataset record count: 2,040,929
Test dataset record count:       2,321,696


### 2. Feature Engineering

In [ ]:
def build_feature_pipeline(
    numeric_cols: list[str],
    categorical_cols: list[str],
    target_col: str = "fare_amount",
) -> Pipeline:
    """Constructs a reusable, configurable PySpark ML feature transformation pipeline.
    """

    sql_transformer = SQLTransformer(
        statement=f"""
        SELECT *,
            CAST(dayofweek(pickup_datetime) AS DOUBLE) AS day_of_week,
            CAST(month(pickup_datetime) AS DOUBLE) AS month,
            CAST(hour(pickup_datetime) AS DOUBLE) AS hour_of_day,
            CASE WHEN dayofweek(pickup_datetime) IN (1, 7) THEN 1.0 ELSE 0.0 END AS is_weekend,
            SIN(2 * 3.141592653589793 * CAST(hour(pickup_datetime) AS DOUBLE) / 24.0) AS sin_hour,
            COS(2 * 3.141592653589793 * CAST(hour(pickup_datetime) AS DOUBLE) / 24.0) AS cos_hour,
            CAST({target_col} AS DOUBLE) AS label
        FROM __THIS__
        """
    )

    imputed_numeric_cols = [f"{col}_imputed" for col in numeric_cols]
    imputer = Imputer(
        inputCols=numeric_cols,
        outputCols=imputed_numeric_cols,
        strategy="median",
    )

    indexed_cols = [f"{col}_indexed" for col in categorical_cols]
    indexer = StringIndexer(
        inputCols=categorical_cols,
        outputCols=indexed_cols,
        handleInvalid="keep", 
    )

    encoded_cols = [f"{col}_vec" for col in categorical_cols]
    encoder = OneHotEncoder(
        inputCols=indexed_cols,
        outputCols=encoded_cols,
        dropLast=True,
    )

    all_feature_cols = (
        imputed_numeric_cols
        + encoded_cols
        + ["day_of_week", "month", "is_weekend", "sin_hour", "cos_hour"]
    )
    assembler = VectorAssembler(
        inputCols=all_feature_cols,
        outputCol="unscaled_features",
        handleInvalid="skip",
    )

    scaler = StandardScaler(
        inputCol="unscaled_features",
        outputCol="features",
        withStd=True,
        withMean=True,
    )

    return Pipeline(
        stages=[
            sql_transformer,
            imputer,
            indexer,
            encoder,
            assembler,
            scaler,
        ]
    )

NUMERIC_FEATURES = ["trip_distance", "temperature_c", "wind_speed_ms", "pm25"]
CATEGORICAL_FEATURES = ["pickup_borough", "dropoff_borough"]

feature_pipeline = build_feature_pipeline(
    numeric_cols=NUMERIC_FEATURES,
    categorical_cols=CATEGORICAL_FEATURES,
    target_col="fare_amount",
)

### 3. Model Training & Evaluation

In [ ]:
from notebooks.evaluate_ml import evaluate_model
from pyspark.ml import Pipeline
from pyspark.ml.regression import GBTRegressor

regressor = GBTRegressor(featuresCol="features", labelCol="label", maxIter=40, maxDepth=6, seed=42)
full_ml_pipeline = Pipeline(stages=[feature_pipeline, regressor])

train_sample = train_raw.sample(withReplacement=False, fraction=0.1, seed=42).cache()

t0 = time.time()
pipeline_model = full_ml_pipeline.fit(train_sample)
print(f"End-to-End ML Pipeline fit completed in {time.time() - t0:.2f} seconds.")

MODEL_SAVE_PATH = str(ROOT / "data" / "models" / "fare_prediction_gbt_pipeline")
pipeline_model.write().overwrite().save(MODEL_SAVE_PATH)
print(f"Trained Pipeline Model saved to: {MODEL_SAVE_PATH}\n")

evaluate_model(model=pipeline_model, test_df=test_raw)

End-to-End ML Pipeline fit completed in 37.32 seconds.
Trained Pipeline Model saved to: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/models/fare_prediction_gbt_pipeline



26/10/01 11:11:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/10/01 11:11:19 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


Inference completed in 0.44 seconds across 2,321,696 rows.



           MODEL EVALUATION SUMMARY               
Root Mean Squared Error (RMSE): $5.5263
Mean Absolute Error (MAE):     $2.6450
Coefficient of Determination (R²): 0.8961

=== Error Breakdown by Pickup Borough ===
+--------------+----------+-----+-----+
|pickup_borough|trip_count|mae  |rmse |
+--------------+----------+-----+-----+
|Manhattan     |2081218   |2.31 |4.24 |
|Queens        |204500    |5.43 |11.89|
|Brooklyn      |28457     |5.46 |8.87 |
|Bronx         |6762      |5.8  |8.44 |
|N/A           |705       |31.96|49.6 |
|Staten Island |54        |18.12|25.17|
+--------------+----------+-----+-----+



DataFrame[taxi_type: string, vendor_id: int, pickup_datetime: timestamp, dropoff_datetime: timestamp, passenger_count: int, trip_distance: double, pickup_location_id: int, pickup_zone: string, pickup_borough: string, dropoff_location_id: int, dropoff_zone: string, dropoff_borough: string, fare_amount: double, tip_amount: double, tolls_amount: double, total_amount: double, temperature_c: double, wind_speed_ms: double, pm25: double, pm25_unit: string, pickup_date: date, pickup_hour: int, day_of_week: double, month: double, hour_of_day: double, is_weekend: decimal(2,1), sin_hour: double, cos_hour: double, label: double, trip_distance_imputed: double, temperature_c_imputed: double, wind_speed_ms_imputed: double, pm25_imputed: double, pickup_borough_indexed: double, dropoff_borough_indexed: double, pickup_borough_vec: vector, dropoff_borough_vec: vector, unscaled_features: vector, features: vector, prediction: double, residual: double, abs_error: double]

### 4. Retraining and inference

In [6]:
loaded_pipeline_model = PipelineModel.load(MODEL_SAVE_PATH)

inference_sample = loaded_pipeline_model.transform(val_raw.limit(5)).select(
    "pickup_datetime", "pickup_borough", "dropoff_borough", "trip_distance", "label", "prediction"
)
print("=== Loaded Model Inference Sample ===")
inference_sample.show(truncate=False)

new_val_sample = val_raw.sample(withReplacement=False, fraction=0.1, seed=42).cache()

retrained_pipeline_model = full_ml_pipeline.fit(new_val_sample)

retrained_save_path = str(ROOT / "data" / "models" / "fare_prediction_gbt_pipeline_v2")
retrained_pipeline_model.write().overwrite().save(retrained_save_path)
print(f"Retrained Pipeline Model successfully fit and saved to: {retrained_save_path}")

=== Loaded Model Inference Sample ===
+-------------------+--------------+---------------+-------------+-----+------------------+
|pickup_datetime    |pickup_borough|dropoff_borough|trip_distance|label|prediction        |
+-------------------+--------------+---------------+-------------+-----+------------------+
|2024-03-09 00:01:33|Manhattan     |Manhattan      |2.9          |16.3 |16.78164944086126 |
|2024-03-09 00:03:03|Manhattan     |Manhattan      |4.7          |23.3 |22.843698378156315|
|2024-03-09 00:03:06|Queens        |Manhattan      |8.8          |35.9 |39.28227629299604 |
|2024-03-09 00:03:37|Queens        |Manhattan      |3.5          |18.4 |19.371096356385678|
|2024-03-09 00:03:44|Manhattan     |Manhattan      |0.5          |4.4  |6.109742977585631 |
+-------------------+--------------+---------------+-------------+-----+------------------+



Retrained Pipeline Model successfully fit and saved to: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/data/models/fare_prediction_gbt_pipeline_v2
